In [61]:
!pip install geopy


Defaulting to user installation because normal site-packages is not writeable


In [76]:
from pymongo import MongoClient

# 1. connect to remote MongoDB
client = MongoClient(
    host="mongo-csgy-6513-spring.db",
    port=27017,
    username="hm3424",
    password="bigdata",
    authSource="bigdata"
)
db = client.bigdata
print("collections:", db.list_collection_names())

for name in db.list_collection_names():
    print(name, db[name].count_documents({}))    


collections: ['sp', 'nikhil', 'ky2684_books_sales', 'restaurants_test_snc8114', 'ms9721_books', 'Q1', 'ashleysun', 'durham_foreclosures', 'ms9721_Q2_restaurants_2', 'ms9721_Q1', 'ky2684_durham_foreclosures', 'ms9721_Q2_foreclosure', 'durham_restaurants', 'juan', 'book_sales', 'test', 'odk_ebb_opinions', 'ky2684_restaurants', 'restaurant', 'restaurants', 'foo', 'ms9721_Q2_restaurants', 'maaz']
sp 2
nikhil 2
ky2684_books_sales 8900
restaurants_test_snc8114 3772
ms9721_books 10000
Q1 0
ashleysun 0
durham_foreclosures 1948
ms9721_Q2_restaurants_2 1758
ms9721_Q1 3772
ky2684_durham_foreclosures 1948
ms9721_Q2_foreclosure 1948
durham_restaurants 1758
juan 6
book_sales 10000
test 3
odk_ebb_opinions 0
ky2684_restaurants 3772
restaurant 3772
restaurants 3772
foo 2
ms9721_Q2_restaurants 2463
maaz 1


In [63]:
restaurants = db.durham_restaurants    # your CSV → should have 2463 docs
foreclosures = db.durham_foreclosures  # your JSON → should have 1948 docs


In [55]:
import os

for root, dirs, files in os.walk(".", topdown=True):
    if "Restaurants_in_Durham_County_NC.csv" in files:
        print("find CSV :", os.path.join(root, "Restaurants_in_Durham_County_NC.csv"))
    if "durham-nc-foreclosure-2006-2016.json" in files:
        print("find JSON :", os.path.join(root, "durham-nc-foreclosure-2006-2016.json"))


find CSV : ./shared/hw4/Restaurants_in_Durham_County_NC.csv
find JSON : ./shared/hw4/durham-nc-foreclosure-2006-2016.json


In [71]:

csv_path = "./shared/hw4/Restaurants_in_Durham_County_NC.csv"
with open(csv_path, "r", encoding="utf-8") as f:
    header = f.readline().strip()
    sample = f.readline().strip()
print("HEADER:", header)
print("SAMPLE ROW:", sample)
print("Fields list:", header.split(","))


HEADER: ID;Premise_Name;Premise_Address1;Premise_Address2;Premise_City;Premise_State;Premise_Zip;Premise_Phone;Hours_Of_Operation;Opening_Date;Closing_Date;Seats;Water;Sewage;Insp_Freq;Est_Group_Desc;Risk;Smoking_Allowed;Type_Description;Rpt_Area_Desc;Status;Transitional_Type_Desc;geolocation
SAMPLE ROW: 56060;WEST 94TH ST PUB;4711 HOPE VALLEY RD;SUITE 6C;DURHAM;NC;27707;(919) 403-0025;;1994-09-01;;60;5 - Municipal/Community;3 - Municipal/Community;4;Full-Service Restaurant;4;NO;1 - Restaurant;Food Service;ACTIVE;FOOD;35.9207272, -78.9573299
Fields list: ['ID;Premise_Name;Premise_Address1;Premise_Address2;Premise_City;Premise_State;Premise_Zip;Premise_Phone;Hours_Of_Operation;Opening_Date;Closing_Date;Seats;Water;Sewage;Insp_Freq;Est_Group_Desc;Risk;Smoking_Allowed;Type_Description;Rpt_Area_Desc;Status;Transitional_Type_Desc;geolocation']


In [77]:
import csv, json
from pymongo import MongoClient

client = MongoClient(
    host="mongo-csgy-6513-spring.db",
    port=27017,
    username="hm3424",
    password="bigdata",
    authSource="bigdata"
)
db = client.bigdata
coll = db.durham_restaurants

# drop old data
coll.drop()

csv_path = "./shared/hw4/Restaurants_in_Durham_County_NC.csv"
docs = []
with open(csv_path, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter=';')
    print("Fieldnames:", reader.fieldnames)
    for row in reader:
        try:

            try:
                seats = int(row.get("Seats",""))
            except:
                seats = None

   
            lat = lon = None
            geo = row.get("geolocation","")
            if "," in geo:
                lat_s, lon_s = geo.split(",",1)
                try:
                    lat = float(lat_s.strip())
                    lon = float(lon_s.strip())
                except:
                    lat = lon = None

     
            doc = { k: row[k] for k in reader.fieldnames }
            doc["Seats"] = seats
    
            doc.pop("geolocation",None)

            if lat is not None and lon is not None:
                doc["geometry"] = {"type":"Point","coordinates":[lon, lat]}
            else:
                doc["geometry"] = None

            docs.append(doc)
        except Exception as e:

            print("Unexpected skip on line", reader.line_num, ":", e)

# bulk insert
if docs:
    coll.insert_many(docs)

# verify
print("Imported documents:", coll.count_documents({}))


Fieldnames: ['ID', 'Premise_Name', 'Premise_Address1', 'Premise_Address2', 'Premise_City', 'Premise_State', 'Premise_Zip', 'Premise_Phone', 'Hours_Of_Operation', 'Opening_Date', 'Closing_Date', 'Seats', 'Water', 'Sewage', 'Insp_Freq', 'Est_Group_Desc', 'Risk', 'Smoking_Allowed', 'Type_Description', 'Rpt_Area_Desc', 'Status', 'Transitional_Type_Desc', 'geolocation']
Imported documents: 2463


In [78]:
print("total docs:", restaurants.count_documents({}))
print("sample doc:", restaurants.find_one())
print("total docs:", foreclosures.count_documents({}))
print("sample doc:", foreclosures.find_one())

total docs: 2463
sample doc: {'_id': ObjectId('68126f2219f295c60dfdd38f'), 'ID': '56060', 'Premise_Name': 'WEST 94TH ST PUB', 'Premise_Address1': '4711 HOPE VALLEY RD', 'Premise_Address2': 'SUITE 6C', 'Premise_City': 'DURHAM', 'Premise_State': 'NC', 'Premise_Zip': '27707', 'Premise_Phone': '(919) 403-0025', 'Hours_Of_Operation': '', 'Opening_Date': '1994-09-01', 'Closing_Date': '', 'Seats': 60, 'Water': '5 - Municipal/Community', 'Sewage': '3 - Municipal/Community', 'Insp_Freq': '4', 'Est_Group_Desc': 'Full-Service Restaurant', 'Risk': '4', 'Smoking_Allowed': 'NO', 'Type_Description': '1 - Restaurant', 'Rpt_Area_Desc': 'Food Service', 'Status': 'ACTIVE', 'Transitional_Type_Desc': 'FOOD', 'geometry': {'type': 'Point', 'coordinates': [-78.9573299, 35.9207272]}}
total docs: 1948
sample doc: {'_id': ObjectId('68126d8ec89535d0e398f367'), 'datasetid': 'foreclosure-2006-2016', 'recordid': '629979c85b1cc68c1d4ee8cc351050bfe3592c62', 'fields': {'parcel_number': '110138', 'geocode': [36.0013755,

In [81]:
from pymongo import MongoClient

# connect
client = MongoClient(
    host="mongo-csgy-6513-spring.db",
    port=27017,
    username="hm3424",
    password="bigdata",
    authSource="bigdata"
)
db = client.bigdata
rest = db.durham_restaurants
fore = db.durham_foreclosures

# ensure 2dsphere indexes
rest.create_index([("geometry", "2dsphere")])
fore.create_index([("geometry", "2dsphere")])

'geometry_2dsphere'

In [82]:
# Q2.1 – compute centroid via aggregation on the server
centroid_doc = rest.aggregate([
    {"$match": {
        "Rpt_Area_Desc": "Food Service",
        "Seats": {"$gte": 40}
    }},
    {"$group": {
        "_id": None,
        "avgLon": {"$avg": {"$arrayElemAt": ["$geometry.coordinates", 0]}},
        "avgLat": {"$avg": {"$arrayElemAt": ["$geometry.coordinates", 1]}}
    }}
]).next()

centroid = (centroid_doc["avgLat"], centroid_doc["avgLon"])
print("Centroid (lat, lon):", centroid)

Centroid (lat, lon): (35.97047224500876, -78.91507069211909)


In [87]:
# Q2.2 – count docs within 2 miles using $geoWithin + $centerSphere
# Earth radius ≈3963.2 miles, so radius in radians = miles/EarthRadius
from pymongo import MongoClient
from geopy.distance import geodesic

# 1. connect
client = MongoClient(
    host="mongo-csgy-6513-spring.db",
    port=27017,
    username="hm3424",
    password="bigdata",
    authSource="bigdata"
)
fore = client.bigdata.durham_foreclosures

# 2. centroid from Q2.1 (lat, lon)
centroid = (35.97047224500876, -78.91507069211909)

# 3. iterate and count
count = 0
for rec in fore.find({}, {"geometry.coordinates": 1, "_id": 0}):
    coords = rec.get("geometry", {}).get("coordinates")
    if not coords or not isinstance(coords, list):
        continue
    lon, lat = coords
    # compute distance
    if geodesic(centroid, (lat, lon)).miles <= 2:
        count += 1

print("Verified foreclosures within 2 miles:", count)


Verified foreclosures within 2 miles: 566
